# Ensemble Voting Test
This notebook demonstrates soft and hard voting with a random image from the dataset.

In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: /home/vertyx/Documents/dermascan


In [2]:
from dataset_manager import DatasetManager
from ensemble_voting import EnsembleVoter, ModelSpec

In [3]:
manager = DatasetManager(data_dir=str(proj_dir / "data"))

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True)

In [ ]:
# Mock model loaders and predictors
class MockModel:
    def __init__(self, probs):
        self.probs = probs

def load_rgb():
    return MockModel([0.2, 0.8])

def load_texture():
    return MockModel([0.6, 0.4])

def predict_proba(model, image):
    return model.probs

specs = [
    ModelSpec(model_id="rgb_resnet50", load=load_rgb, predict_proba=predict_proba),
    ModelSpec(model_id="texture_resnet18", load=load_texture, predict_proba=predict_proba),
]

### Soft Voting

In [5]:
voter = EnsembleVoter(
    model_specs=specs,
    strategy="soft",
    weights={"rgb_resnet50": 0.6, "texture_resnet18": 0.4},
    tie_breaker="max_prob",
    label_names=["nevus", "melanoma"],
)

_ = voter.run(image="dummy_image")

Ensemble voting summary
Strategy: soft
Branch outputs:
- rgb_resnet50: label=melanoma, confidence=0.8000, probs=[0.2, 0.8]
- texture_resnet18: label=nevus, confidence=0.6000, probs=[0.6, 0.4]
Final: label=melanoma, confidence=0.6400, probs=[0.36, 0.64]


### Hard Voting

In [6]:
# Hard voting with a tie breaker
voter_hard = EnsembleVoter(
    model_specs=specs,
    strategy="hard",
    tie_breaker="max_prob",
    label_names=["nevus", "melanoma"],
)

_ = voter_hard.run(image="dummy_image")

Ensemble voting summary
Strategy: hard
Branch outputs:
- rgb_resnet50: label=melanoma, confidence=0.8000, probs=[0.2, 0.8]
- texture_resnet18: label=nevus, confidence=0.6000, probs=[0.6, 0.4]
Final: label=melanoma, confidence=0.5000, probs=[0.5, 0.5]
